# Porseman question filtering

This notebook reproduces `scripts/filter_porseman_questions.py` one filter at a time. Each filter cell prints row counts and shows examples.

## Filter types

1. **Question-mark validation**: removes a row when `question` does not end with Persian `؟` or English `?`. Set `QUESTION_MARK_POSITION = "any"` to accept a question mark anywhere in the text.
2. **Answer copied into question**: removes a row when its normalized question exactly matches any normalized `content_text` value. Normalization unifies Persian/Arabic letter forms, invisible spacing, whitespace, and case.
3. **Extremely long question**: removes a row when its normalized question has at least 1,200 characters.
4. **Suspicious article or answer text**: marks six independent signals: question length at least 500; question at least 300 characters and twice the answer length; article phrases; at least three citations; at least four numbered-list items; or at least three newlines. A row is removed from this category only when it has three or more of these signals.

## Setup

Configure the input and output paths, question-mark rule, and the number of rejected examples to display.

In [1]:
from __future__ import annotations

import csv
import html
import re
import sys
from pathlib import Path
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "filter_porseman_questions.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "filter_porseman_questions.py").is_file():
    raise FileNotFoundError("Run this notebook from the project or its notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.filter_porseman_questions import (
    ARTICLE_MARKERS,
    has_required_question_mark,
    normalize,
)

INPUT_PATH = PROJECT_ROOT / "data/raw/porseman_clean.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/porseman_questions_filtered.csv"
REJECTED_PATH = PROJECT_ROOT / "data/processed/porseman_questions_rejected.csv"
QUESTION_MARK_POSITION = "end"  # Change to "any" to allow a mark anywhere.
SAMPLE_SIZE = 5
RETAINED_ID_COLUMN = "id"
BGE_TOKENIZER_NAME = "BAAI/bge-m3"
PASSAGE_TOKEN_LIMIT = 1024
TOKENIZER_BATCH_SIZE = 128
TOKEN_COUNT_OUTPUT_PATH = PROJECT_ROOT / "data/processed/porseman_retained_answer_token_counts.csv"


## Load data

Load the CSV, validate its required columns, and prepare normalized answers for the filters below.

In [2]:
with INPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    fieldnames = list(reader.fieldnames or ())
    missing = {"question", "content_text"}.difference(fieldnames)
    if missing:
        raise ValueError(f"Missing required CSV columns: {', '.join(sorted(missing))}")
    all_rows = [{**row, "_row_number": index, "_reasons": []} for index, row in enumerate(reader, start=2)]

active_rows = all_rows.copy()
normalized_answers = {normalize(row["content_text"]) for row in all_rows if normalize(row["content_text"])}

def short(value, limit=240):
    value = (value or "").replace("\n", " ")
    return value if len(value) <= limit else value[:limit] + "..."

def show_examples(rows, title):
    print(f"{title}: {len(rows):,} row(s)")
    if not rows:
        return
    cells = ["<tr><th>CSV row</th><th>Question</th><th>Reasons / signals</th></tr>"]
    for row in rows[:SAMPLE_SIZE]:
        reasons = row.get("_reasons", []) or row.get("_signals", [])
        cells.append("<tr><td>{}</td><td>{}</td><td>{}</td></tr>".format(
            row["_row_number"], html.escape(short(row["question"])), html.escape(", ".join(reasons))
        ))
    display(HTML("<table style='width:100%; text-align:left'>" + "".join(cells) + "</table>"))

print(f"Rows before filtering: {len(active_rows):,}")


Rows before filtering: 47,363


## Filters that remove rows immediately

### 1. Question-mark validation

A valid question must end with `؟` or `?` by default. This prevents titles, fragments, and answer text without a question mark from entering the dataset.

In [3]:
before = len(active_rows)
removed = [row for row in active_rows if not has_required_question_mark(row["question"] or "", QUESTION_MARK_POSITION)]
for row in removed:
    row["_reasons"].append("missing_question_mark")
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Missing question mark: before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for missing_question_mark")


Missing question mark: before=47,363, removed=14,970, after=32,393
Examples removed for missing_question_mark: 14,970 row(s)


CSV row,Question,Reasons / signals
6,با سلام بارها پرسیدم که لیله المبیت در کدوم کتاب های تاریخی و صحاح سته اومده با ذکر منبع جلد و صفحه ... اما بزرگواران درست به من جواب نمیدن ... لطف بفرمایید ممنون میشم,missing_question_mark
8,"بزرگان جامعه شعار جذب حداکثری و دفع حداقلی می دهند اما در عمل برعکس است. شخصیتها, احزاب, روزنامه ها که اندک انتقادی می کنند بلافاصله برچسب ضد نظام می خورند و دچار محرومیتهای متنوع می شوند. این یعنی اشکال ساختاری در سیاست و نافهمی سیاسی و نا...",missing_question_mark
9,رفتار امام رضا(ع) چگونه بود؟ (علمی.فرهنگی و...),missing_question_mark
37,سلام ببخشید به لوازم ارایشی که در منزل استفاده میشه خمس تعلق میگیره؟ممنون,missing_question_mark
41,«کن فیکون» چگونه ممکن است؟ یعنی من عقلم دیگه جوابگو نیست همش می بینم ما با عالم ماده کار می کنیم و ... خدا هم که از ماده استفاده نمی کنه، پس چگونه به یک چیز مادی گفته «باش» و موجود میشه؟ اصلا با عقل سازگار نیست، مگه میشه به یک چیزی بگی باش ...,missing_question_mark


### 2. Answer copied into question

This removes rows whose question is exactly the same as any answer after text normalization. It catches answers accidentally written in the `question` column.

In [4]:
before = len(active_rows)
removed = [row for row in active_rows if normalize(row["question"]) in normalized_answers]
for row in removed:
    row["_reasons"].append("question_matches_an_answer")
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Question matches an answer: before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for question_matches_an_answer")


Question matches an answer: before=32,393, removed=0, after=32,393
Examples removed for question_matches_an_answer: 0 row(s)


### 3. Extremely long question

Questions with 1,200 or more normalized characters are removed directly. They are likely to be article or answer text instead of a user question.

In [5]:
before = len(active_rows)
removed = [row for row in active_rows if len(normalize(row["question"])) >= 1200]
for row in removed:
    row["_reasons"].append("very_long_question")
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Very long question (>= 1200): before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for very_long_question")


Very long question (>= 1200): before=32,393, removed=41, after=32,352
Examples removed for very_long_question: 41 row(s)


CSV row,Question,Reasons / signals
36,سوره زمر، آیه۷۱: وَسِیقَ الَّذِینَ کَفَرُوا إِلَى جَهَنَّمَ زُمَرًا حَتَّى إِذَا جَاءُوهَا فُتِحَتْ أَبْوَابُهَا وَقَالَ لَهُمْ خَزَنَتُهَا أَلَمْ یَأْتِکُمْ رُسُلٌ مِنْکُمْ یَتْلُونَ عَلَیْکُمْ آیَاتِ رَبِّکُمْ وَیُنْذِرُونَکُمْ لِقَاءَ یَ...,very_long_question
761,سلام. من دختری ۲۸ ساله و مجرد هستم. خانواده ای مقید و بدون هیچ حاشیه. هم توو اجتماع هستم، هم دانشجوی دکتری در یکی از دانشگاه های مطرح هستم،هم دارای ارتباط گسترده با اقوام و دوستان و اشنایان. هم ظاهر زیبایی دارم هم از لحاظ شخصیت به پختگی رسی...,very_long_question
1158,آیا میشه گفت که اصحاب کهف به مدت مثلا سیصد و خورده ای سال در زمان سفر کردند و جلو رفتند نه اینکه مثلا سیصد و خورده ای سال با زمان خودمان در غار خواب بوده اند؟یعنی طبق ایه ۱۹ سوره کهف وقتی اصحاب کهف از خواب بیدار شدند٬ دقیقا نمیدونن چقدر خوا...,very_long_question
1449,با سلام و احترام شبهه ی مطروح در ذیل مورد توجه دشمنان بوده است. بنده قصد جسارت به ساحت امام علی (ع) را نداشته و صرفا برای روشن تر شدن موضوع و نزدیک شدن به اندیشه های امام علی (ع) این سوال را می پرسم. خواهشمندم پاسخ را به نحوی اقناعی و مکفی ...,very_long_question
1789,سلام. تو قسمت اخرین مطالب سایتتون تاریخ ۵۹۵۹ و سقوط اسرائیل هست. این رو هم کپی کردم از روی سوال اون شخص و پاسخی که بهش داده شد: سلام علیکم سال ۵۹۵۹ عبری قمری به سال شمسی ما از چه تاریخی شروع و چه تاریخی تمام میشود ؟ جواب اجمالی : در باره تق...,very_long_question


### 4. Suspicious article or answer text

The next six cells check signals that suggest an article or an answer was placed in `question`. A single signal does not remove a row; the final cell removes a row only when at least three signals occur together.

Normalize question and answer text, calculate their lengths, and reset the suspicious-signal list for each remaining row.

In [6]:
for row in active_rows:
    row["_normalized_question"] = normalize(row["question"])
    row["_question_length"] = len(row["_normalized_question"])
    row["_answer_length"] = len(normalize(row["content_text"]))
    row["_signals"] = []


Mark questions with at least 500 normalized characters; this signal alone does not remove a row.

In [7]:
for row in active_rows:
    if row["_question_length"] >= 500:
        row["_signals"].append("long_question")
matches = [row for row in active_rows if "long_question" in row["_signals"]]
show_examples(matches, "long_question candidates (>= 500 characters)")


long_question candidates (>= 500 characters): 494 row(s)


CSV row,Question,Reasons / signals
110,وقتی حاجتی داریم به ائمه معصومین یا بندگان ناب خدا متوسل میشیم. سوال من این است که ممکن است به خاطر اینکه یکی از امامان چون بیشتر نسبت به ما عنایت دارند قطعا حاجتمون رو بدن و در مورد امام یا اولیای الهی دیگر چون کمتر عنایت و دوستمون دارن، ک...,long_question
127,باسلام و شب بخیر ، ضمن تسلیت ایام اربعین حسینی (ع) سوالاتی دارم که امیدوارم ان را کامل برایم شرح دهید. ۱. در حدیثی امده هست که درجه ای نزد خداوند است که هیچ بنده ای به ان نمی رسد مگر با دعا ، خب این حدیث به چه معناست ، یعنی باید برای رسیدن ...,long_question
293,برخی مانند هیوم در باره برهان نظم میگویند مقایسه میان خلقت جهان و اشباع که مثلا نظم در یک ساعت را مثال میزنیم و به نظم دنیا شبیهش می‌کنیم بسیار سست و ضعیف است زیرا چنین مقایسه ای تنها میان دو پدیدهٔ تجربی اعتبار دارد، نه میان یک پدیدهٔ تجرب...,long_question
308,باسلام و خسته نباشید یه سوال داشتم اگه مثلا یک شخصی از لحاظ اعتقادی و فکری ضعف داشته باشه یا حالا اعتقادات باطل رو داشته باشه(مثلا انکار کردن دین و پیامبر و یا روز قیامت و افکارش در موضوعات مختلف بر جبهه باطل باشه) ایا اگر مثلا خلقیات و رفت...,long_question
316,حاج آقا سلام و ممنون از وقتی که لطف کردید،میتوانم عرض کنم که در واقع سؤال من این هست که مبدءمحاسبه ۱۳۵۰متر،درمورد شهرهایی که خروجی آنهابوسیله جاده های کمربندی است که مسافت قابل توجهی رادرعرض وموازات شهر،طی میشود،کجاست؟یعنی مسافر از شهرخارج ...,long_question


Mark questions that are at least 300 characters and at least twice as long as their answer.

In [8]:
for row in active_rows:
    if row["_question_length"] >= 300 and row["_question_length"] >= max(1, row["_answer_length"]) * 2:
        row["_signals"].append("question_much_longer_than_answer")
matches = [row for row in active_rows if "question_much_longer_than_answer" in row["_signals"]]
show_examples(matches, "question_much_longer_than_answer candidates")


question_much_longer_than_answer candidates: 32 row(s)


CSV row,Question,Reasons / signals
74,خواستم بدونم اگر در دلمون به یکی فحش دهیم یا در تنهایی وقتی کسی اصلا نیست خود طرقی که بهش فحش هم میدیم نیست اگر فحش بدهیم چه حکمی دارد؟اگر در تنهایی پشت سر کسی غیبت کنیم یعنی کسی جز خودمون متوجه نشود حکم چیست؟و اینکه اگر در تنهایی به مادر ک...,question_much_longer_than_answer
276,سلام بنده مقلد حضرت امام خمینی بودم مدتی است عمل جراحی پروستات انجام داده ام و بصورت کامل پروستات و کیسه منی تخلیه شده است. در حال حاضر در اثر تحریک جنسی هیچگونه مایع ای از ناحیه مجرای ادرار خارج نمیشود و فقط حس انزال وجود دارد و طبق گفته د...,question_much_longer_than_answer
316,حاج آقا سلام و ممنون از وقتی که لطف کردید،میتوانم عرض کنم که در واقع سؤال من این هست که مبدءمحاسبه ۱۳۵۰متر،درمورد شهرهایی که خروجی آنهابوسیله جاده های کمربندی است که مسافت قابل توجهی رادرعرض وموازات شهر،طی میشود،کجاست؟یعنی مسافر از شهرخارج ...,"long_question, question_much_longer_than_answer"
391,اگه درکل اصلا کاری به قبل از وقت یا داخل وقت غسل کردن نداشته باشیم فقط هر زمانی که غسل کردیم، غسل رو برای طهارت انجام بدیم قربه الی الله وتوجهی به واجب ومستحب بودنش نکنیم طوری که اگه ازمون پرسیدن چکار میکنی بگیم برای طهارت از مثلا جنابت غسل...,question_much_longer_than_answer
862,سلام،درموردنمازوروزه میت،سوال داشتم:شخصی درسال گذشته سه سال نمازوروزه قبول کرده که بخواندوبگیرد،وباایشان برای نه سال،کلا،نه میلیون تومان قرارگذاشتن که بدهند،رقمی که قرارگذاشتند،تقریبا،نصف متعارف قیمت ان بوده،ولی ایشان مطلع نبوده و قبول کرده...,question_much_longer_than_answer


Mark questions containing article-style phrases such as a reference list, keywords, or footnotes.

In [9]:
for row in active_rows:
    if any(marker in row["_normalized_question"] for marker in ARTICLE_MARKERS):
        row["_signals"].append("article_phrase")
matches = [row for row in active_rows if "article_phrase" in row["_signals"]]
show_examples(matches, "article_phrase candidates")


article_phrase candidates: 21 row(s)


CSV row,Question,Reasons / signals
1983,با سلام۱- چه دلیلی بر ختم نبوت توسط پیامبر(ص) وجود دارد؟ ۲- ایا اعتقاد به ۱۲ امام بعد از پیامبر با ختم نبوت در تناقض نیست؟؟ چرا که امامان هم مانند پیامبران دارای علم کامل به دین و شریعت و معصوم هستند، وظیفه دارند مردم را هدایت کنند و دین را...,article_phrase
2456,سلام.آیابعدازظهورروزقیامت خواهدشدویازندگی ادامه دارد؟,article_phrase
3093,در یکی از درسهای کتاب عربی دوره دبیرستان خوانده بودیم که آسیه همسر فرعون با سنگ بزرگی که از ارتفاع بر روی بدنش انداختن به شهادت رسید ولی در این مقاله نوشته شده است که یه قصاب تمام پوست بدنش را کند این تناقضات را چگونه قابل توجیه هستند؟,article_phrase
3308,سلام ایا در روز قیامت ، هر کس در سن خودش محشور میشود؟ ایا در بهشت ، همه جوان میشوند ؟ زندگی در ان دنیا تا کی ادامه دارد؟,article_phrase
7403,دوست محترم در این باره مفید به نظر می رسد که شناخت خود را نسبت به قرآن افزایش دهید.برای شناخت قرآن و بهره مندی بیشترنیز باید مراحلی را طی کرد از آن جمله : ۱. آشنایی با اهداف قرآن و محورهای اساسی آن . ۲. آشنایی با ابزار و شرایط فهم قرآن ۳. ر...,"long_question, article_phrase"


Mark questions containing at least three bracketed numeric citations, such as `[1]`.

In [10]:
citation_pattern = re.compile(r"\[\s*[\u06F0-\u06F90-9]+\s*\]")
for row in active_rows:
    if len(citation_pattern.findall(row["_normalized_question"])) >= 3:
        row["_signals"].append("many_citations")
matches = [row for row in active_rows if "many_citations" in row["_signals"]]
show_examples(matches, "many_citations candidates (>= 3)")


many_citations candidates (>= 3): 0 row(s)


Mark questions containing at least four numbered-list items.

In [11]:
numbered_list_pattern = re.compile(r"(?:^|\s)[\u06F0-\u06F90-9]+[.)\u0600-]")
for row in active_rows:
    if len(numbered_list_pattern.findall(row["_normalized_question"])) >= 4:
        row["_signals"].append("numbered_list")
matches = [row for row in active_rows if "numbered_list" in row["_signals"]]
show_examples(matches, "numbered_list candidates (>= 4)")


numbered_list candidates (>= 4): 25 row(s)


CSV row,Question,Reasons / signals
127,باسلام و شب بخیر ، ضمن تسلیت ایام اربعین حسینی (ع) سوالاتی دارم که امیدوارم ان را کامل برایم شرح دهید. ۱. در حدیثی امده هست که درجه ای نزد خداوند است که هیچ بنده ای به ان نمی رسد مگر با دعا ، خب این حدیث به چه معناست ، یعنی باید برای رسیدن ...,"long_question, numbered_list"
921,سه تا سول داشتم. ۱.شبکه ملی اطلاعات چیست؟ ۲.ایا درایران شبکه ی ملی اطلاعات وجو دارد؟ ۳.اگرنه پس چرا شبکه ملی اطلاعات تشکیل نشده است؟ ۴.در کشور های توسعه یافته شبکه ی ملی اطلاعات وجود دارد؟ به چه شکلی؟,numbered_list
1844,درباره رفتن به درب خانه اصحاب برای اتمام حجت و یاداوری پیام غدیر چند پرسش دارم: ۱. این رفتن به چه کیفیتی و چند بار بوده و تعداد صحابه چند نفر بوده است؟ ۲. در نهایت چه کسانی حاضر شدند با مولا بیعت کنند و ابوبکر را خلیفه محسوب نکردند؟ ۳. تراش...,numbered_list
4818,سلام .اعمالی که مانع استجابت دعا می‌شوند عبارتند از: ۱. گناه ۲. استفاده از مال حرام ۳. ستم کردن به مردم و کمک کردن به ستم‌کاران ۴. نافرمانی از پدر و مادر و ازردن ان‌ها ۵. بریدن از خویشاوندان ۶. ترک امر به معروف و نهی از منکر ۷. سبک شمردن نم...,numbered_list
5550,‌سلام جناب استاد احد ایمدرباره خلق ادم و حوا.۱.ظاهرا در سه جای قران این عبارت امده که خلقکم من نفس واحده،مراد از نفس واحده دقیقا چیست؟ ۲.در دو جا به جای خلق جعل دارد چه مطلبی را میخواهد برساند؟ ۳.منها زوجها دقیقا چه معنایی را می رساند ایا غ...,numbered_list


Mark questions with at least three newline characters, which usually indicates several paragraphs of article text.

In [12]:
for row in active_rows:
    if (row["question"] or "").count("\n") >= 3:
        row["_signals"].append("many_paragraphs")
matches = [row for row in active_rows if "many_paragraphs" in row["_signals"]]
show_examples(matches, "many_paragraphs candidates (>= 3 newlines)")


many_paragraphs candidates (>= 3 newlines): 0 row(s)


Remove rows only when at least three suspicious signals were marked, preserving ordinary long questions with fewer signals.

In [13]:
before = len(active_rows)
removed = [row for row in active_rows if len(row["_signals"]) >= 3]
for row in removed:
    row["_reasons"].extend(row["_signals"])
removed_ids = {id(row) for row in removed}
active_rows = [row for row in active_rows if id(row) not in removed_ids]
print(f"Three or more suspicious signals: before={before:,}, removed={len(removed):,}, after={len(active_rows):,}")
show_examples(removed, "Examples removed for suspicious text")


Three or more suspicious signals: before=32,352, removed=1, after=32,351
Examples removed for suspicious text: 1 row(s)


CSV row,Question,Reasons / signals
7403,دوست محترم در این باره مفید به نظر می رسد که شناخت خود را نسبت به قرآن افزایش دهید.برای شناخت قرآن و بهره مندی بیشترنیز باید مراحلی را طی کرد از آن جمله : ۱. آشنایی با اهداف قرآن و محورهای اساسی آن . ۲. آشنایی با ابزار و شرایط فهم قرآن ۳. ر...,"long_question, article_phrase, numbered_list"


## BGE-M3 answer token lengths

This checks answers retained after all question filters with the `BAAI/bge-m3` tokenizer used for training. Token counts include special tokens, so they can be compared directly with a 1,024-token passage limit.

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BGE_TOKENIZER_NAME)
answer_rows = [row for row in active_rows if (row["content_text"] or "").strip()]
empty_answer_count = len(active_rows) - len(answer_rows)
if not answer_rows:
    raise ValueError("No non-empty answers remain after filtering.")

for start in range(0, len(answer_rows), TOKENIZER_BATCH_SIZE):
    batch = answer_rows[start : start + TOKENIZER_BATCH_SIZE]
    encoded = tokenizer([row["content_text"] for row in batch], add_special_tokens=True, padding=False, truncation=False)
    for row, token_ids in zip(batch, encoded["input_ids"]):
        row["_answer_token_count"] = len(token_ids)

answer_token_counts = sorted(row["_answer_token_count"] for row in answer_rows)
print(f"Tokenizer: {BGE_TOKENIZER_NAME}")
print(f"Non-empty retained answers analyzed: {len(answer_rows):,}")
print(f"Empty retained answers excluded: {empty_answer_count:,}")


c:\Users\1\Documents\finetune-embedding-models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (9355 > 8192). Running this sequence through the model will result in indexing errors


Tokenizer: BAAI/bge-m3
Non-empty retained answers analyzed: 32,351
Empty retained answers excluded: 0


In [15]:
def percentile(values, percent):
    position = (len(values) - 1) * percent / 100
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    return round(values[lower] + (values[upper] - values[lower]) * (position - lower))

within_limit = sum(count <= PASSAGE_TOKEN_LIMIT for count in answer_token_counts)
token_count_rows = sorted(answer_rows, key=lambda row: row["_answer_token_count"], reverse=True)
over_limit_rows = [row for row in token_count_rows if row["_answer_token_count"] > PASSAGE_TOKEN_LIMIT]

print(f"Passage limit: {PASSAGE_TOKEN_LIMIT:,} tokens")
print(f"At or below limit: {within_limit:,} / {len(answer_rows):,} ({within_limit / len(answer_rows):.2%})")
print(f"Above limit: {len(over_limit_rows):,} / {len(answer_rows):,} ({len(over_limit_rows) / len(answer_rows):.2%})")
print(f"Min / median / p90 / p95 / p99 / max: {answer_token_counts[0]:,} / {percentile(answer_token_counts, 50):,} / {percentile(answer_token_counts, 90):,} / {percentile(answer_token_counts, 95):,} / {percentile(answer_token_counts, 99):,} / {answer_token_counts[-1]:,}")

table_rows = ["<tr><th>CSV row</th><th>Tokens</th><th>Question</th><th>Answer</th></tr>"]
for row in over_limit_rows[:SAMPLE_SIZE]:
    table_rows.append("<tr><td>{}</td><td>{:,}</td><td>{}</td><td>{}</td></tr>".format(row["_row_number"], row["_answer_token_count"], html.escape(short(row["question"])), html.escape(short(row["content_text"]))))
display(HTML("<h4>Examples above the token limit</h4><table style='width:100%; text-align:left'>" + "".join(table_rows) + "</table>"))

TOKEN_COUNT_OUTPUT_PATH = globals().get(
    "TOKEN_COUNT_OUTPUT_PATH",
    PROJECT_ROOT / "data/processed/porseman_retained_answer_token_counts.csv",
)
TOKEN_COUNT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with TOKEN_COUNT_OUTPUT_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=["csv_row_number", "answer_token_count", "question", "content_text"])
    writer.writeheader()
    writer.writerows({
        "csv_row_number": row["_row_number"],
        "answer_token_count": row["_answer_token_count"],
        "question": row["question"],
        "content_text": row["content_text"],
    } for row in token_count_rows)

print(f"Per-answer token counts: {TOKEN_COUNT_OUTPUT_PATH}")


Passage limit: 1,024 tokens
At or below limit: 17,371 / 32,351 (53.70%)
Above limit: 14,980 / 32,351 (46.30%)
Min / median / p90 / p95 / p99 / max: 24 / 909 / 3,578 / 4,712 / 6,716 / 21,245


CSV row,Tokens,Question,Answer
7508,"21,245",در یوم الشورا طبق نقل گفته شده حضرت علی (ع) برای خود هفتاد خصیصه قایل شدند که در هیچ مومن دیگری وجود ندارد این نقل حضرت امیر(ع) درکدام کتاب میتوان یافت ؟,حضرت على(علیه السلام) در یکى از مناشده (سوگندیه)هاى خود در روز شورا (شوراى شش نفره عمر) به سال 23 یا آغاز سال 24 از هجرت، ابتدا از فضایل و امتیازهایى که آن حضرت و خاندانش به آن ها مفتخر بوده سخن به میان آورد و با حاضران با قید سوگند، احتجاج...
26235,"14,621",یا ایهاالذین آمنوا در قرآن کریم چند بار تکرار شده است ؟,جمله یا ایهاالذین آمنوا 89 بار در قران ذکر شده که عبارتند از :البقرة : 104 یا أَیُّهَا الَّذینَ آمَنُوا لا تَقُولُوا راعِنا وَ قُولُوا انْظُرْنا وَ اسْمَعُوا وَ لِلْکافِرینَ عَذابٌ أَلیمٌ البقرة : 153 یا أَیُّهَا الَّذینَ آمَنُوا اسْتَعینُو...
20025,"11,767",نشانه های شیعیان واقعی امام علی(ع) که در روایات آمده است چیست؟,اوصاف شیعه در نگاه اهل بیت علیهم السلام پیشگفتار: آنچه در پیش رو است، مجموعه اى است فشرده درباره فرهنگ و اوصاف شیعه که در پرتو گردآورى روایات نقل شده از امامان معصوم علیهم السلام تنظیم و ترتیب یافته است. این نوشته که ترسیم کننده سیماى شیعه ...
28369,"11,585",آیا آیات و احادیثی دربارۀ وصیّ و خلیفۀ پیامبر اسلام(ص) از منابع اهل سنّت وجود دارد؟,آیات و احادیثی دربارۀ وصیّ و خلیفۀ پیامبر اسلام(ص) از منابع اهل سنّت آیۀ ولایت:«إنما ولیُّکم الله و رسوله و الذین امنوا الذین یقیمون الصلاه و یؤتون الزکاه و هم راکعون...جز این نیست که ولیّ و سرپرست شما خداست و رسول او و مؤمنینی که نماز بر پ...
16527,"10,769",آیا از امام صادق علیه السلام یا یکی از شاگردان درجه یک ایشان مانند دانشمندان امروزی درباره متفکران و دانشمندان قبل از خودشان(نظیر ارشمیدس، افلاطون و ...) نظریه یا ردّیه ای وجود دارد؟,تفاوت اساسی بین علم الهی و علم بشری وجود دارد. منشأ علوم انبیا علیهم السلام بوده اند. مواردی نیز در زیر تقدیم میگردد: 1ـ از بین شاگردان آن حضرت، جابر بن حیّان، تحقیقات زیادی در علوم داشته و بسیاری از نظرات گذشتگان را ردّ نموده و برخی را نیز...


Per-answer token counts: C:\Users\1\Documents\finetune-embedding-models\data\processed\porseman_retained_answer_token_counts.csv


## Assign IDs to retained rows

Each remaining row receives a stable ID based on its original CSV row number. This ID is added only to the filtered output.

In [16]:
RETAINED_ID_COLUMN = globals().get("RETAINED_ID_COLUMN", "id")
for row in active_rows:
    row[RETAINED_ID_COLUMN] = f"porseman-{row['_row_number']}"

print(f"Assigned {len(active_rows):,} stable IDs to retained rows.")
id_samples = ["<tr><th>ID</th><th>Question</th></tr>"]
for row in active_rows[:SAMPLE_SIZE]:
    id_samples.append("<tr><td>{}</td><td>{}</td></tr>".format(html.escape(row[RETAINED_ID_COLUMN]), html.escape(short(row["question"]))))
display(HTML("<table style='width:100%; text-align:left'>" + "".join(id_samples) + "</table>"))


Assigned 32,351 stable IDs to retained rows.


ID,Question
porseman-2,راهکارهای ایجاد وحدت مستند به آیات قرآن چیست؟
porseman-3,در خصوص آشنایی با زندگی امام حسن عسکری (علیه‌السلام) توضیحاتی می خواستم؟
porseman-4,امام موسی صدر که بود؟ چرا لقب امام به ایشان داده ایم؟
porseman-5,مهمترین کرامت امام عسکرى(ع) چى بوده؟
porseman-7,چرا مذهب شیعه را مذهب جعفری می گویند و از امام صادق(علیه السلام) به عنوان رئیس و بنیان گذار مذهب یاد می کنند؟


## Final counts and CSV output

In [17]:
rejected_rows = [row for row in all_rows if row["_reasons"]]
print(f"Rows before filtering: {len(all_rows):,}")
print(f"Rows after filtering:  {len(active_rows):,}")
print(f"Rows removed:          {len(rejected_rows):,}")
show_examples(rejected_rows, "All rejected-row examples")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
filtered_fieldnames = [RETAINED_ID_COLUMN, *fieldnames]
with OUTPUT_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=filtered_fieldnames)
    writer.writeheader()
    writer.writerows({name: row.get(name, "") for name in filtered_fieldnames} for row in active_rows)

with REJECTED_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=[*fieldnames, "rejection_reason"])
    writer.writeheader()
    writer.writerows({**{name: row[name] for name in fieldnames}, "rejection_reason": ";".join(row["_reasons"])} for row in rejected_rows)

print(f"Filtered CSV: {OUTPUT_PATH}")
print(f"Rejected CSV: {REJECTED_PATH}")


Rows before filtering: 47,363
Rows after filtering:  32,351
Rows removed:          15,012
All rejected-row examples: 15,012 row(s)


CSV row,Question,Reasons / signals
6,با سلام بارها پرسیدم که لیله المبیت در کدوم کتاب های تاریخی و صحاح سته اومده با ذکر منبع جلد و صفحه ... اما بزرگواران درست به من جواب نمیدن ... لطف بفرمایید ممنون میشم,missing_question_mark
8,"بزرگان جامعه شعار جذب حداکثری و دفع حداقلی می دهند اما در عمل برعکس است. شخصیتها, احزاب, روزنامه ها که اندک انتقادی می کنند بلافاصله برچسب ضد نظام می خورند و دچار محرومیتهای متنوع می شوند. این یعنی اشکال ساختاری در سیاست و نافهمی سیاسی و نا...",missing_question_mark
9,رفتار امام رضا(ع) چگونه بود؟ (علمی.فرهنگی و...),missing_question_mark
36,سوره زمر، آیه۷۱: وَسِیقَ الَّذِینَ کَفَرُوا إِلَى جَهَنَّمَ زُمَرًا حَتَّى إِذَا جَاءُوهَا فُتِحَتْ أَبْوَابُهَا وَقَالَ لَهُمْ خَزَنَتُهَا أَلَمْ یَأْتِکُمْ رُسُلٌ مِنْکُمْ یَتْلُونَ عَلَیْکُمْ آیَاتِ رَبِّکُمْ وَیُنْذِرُونَکُمْ لِقَاءَ یَ...,very_long_question
37,سلام ببخشید به لوازم ارایشی که در منزل استفاده میشه خمس تعلق میگیره؟ممنون,missing_question_mark


Filtered CSV: C:\Users\1\Documents\finetune-embedding-models\data\processed\porseman_questions_filtered.csv
Rejected CSV: C:\Users\1\Documents\finetune-embedding-models\data\processed\porseman_questions_rejected.csv
